In [1]:
!pip install cfbd

  Using cached cfbd-5.13.2-py3-none-any.whl.metadata (737 bytes)
  Using cached pydantic-1.10.26-cp311-cp311-manylinux2014_x86_64.manylinux_2_17_x86_64.whl.metadata (155 kB)
  Using cached aenum-3.1.17-py3-none-any.whl.metadata (3.8 kB)
Using cached cfbd-5.13.2-py3-none-any.whl (245 kB)
Using cached pydantic-1.10.26-cp311-cp311-manylinux2014_x86_64.manylinux_2_17_x86_64.whl (2.8 MB)
Using cached aenum-3.1.17-py3-none-any.whl (165 kB)
  Attempting uninstall: pydantic
    Found existing installation: pydantic 2.8.2
    Uninstalling pydantic-2.8.2:
      Successfully uninstalled pydantic-2.8.2
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
pydantic-settings 2.13.1 requires pydantic>=2.7.0, but you have pydantic 1.10.26 which is incompatible.
nflreadpy 0.1.5 requires pydantic>=2.0.0, but you have pydantic 1.10.26 which is incompatible.
jupyterhub 5.1.0 requires py

In [132]:
import time
import os
import cfbd
from cfbd.models.player_stat import PlayerStat
from cfbd.models.season_type import SeasonType
from cfbd.rest import ApiException
from pprint import pprint

# Defining the host is optional and defaults to https://api.collegefootballdata.com
# See configuration.py for a list of all supported configuration parameters.
configuration = cfbd.Configuration(
    host = "https://api.collegefootballdata.com"
)

# The client must configure the authentication and authorization parameters
# in accordance with the API server security policy.
# Examples for each auth method are provided below, use the example that
# satisfies your auth use case.

# Configure Bearer authorization: apiKey
configuration = cfbd.Configuration(
    access_token = "ph862V5wdd7MjVNjmVkvfAo/bNXL3YgnSR9jUGm7lLUffAnAag9s6tCwCBF9fOx2"
)

raw_data = None

# Enter a context with an instance of the API client
with cfbd.ApiClient(configuration) as api_client:
    # Create an instance of the API class
    api_instance = cfbd.StatsApi(api_client)
    year = 2019
    team = "Alabama"

    try:
        api_response = api_instance.get_player_season_stats(year, team=team)#, conference=conference, team=team, start_week=start_week, end_week=end_week, season_type=season_type, category=category)
        print(api_response[0])
        # print("The response of StatsApi->get_player_season_stats:\n")
        # pprint(api_response)
        raw_data = api_response
    except Exception as e:
        print("Exception when calling StatsApi->get_player_season_stats: %s\n" % e)

season=2019 player_id='3925350' player='Anfernee Jennings' position='LB' team='Alabama' conference='SEC' category='defensive' stat_type='PD' stat='5'


In [3]:
import pandas as pd

def parse_cfbd_player_stats(stats_list, player_name):
    """
    Parses all of a players stats into a single row of stats

    Args:
        stats_list: List of PlayerStat objects returned from cfbd API
        player_name: String name of the player to filter for

    Returns:
        A pandas DataFrame with the player's stats in wide format
    """

    player_stats = [s for s in stats_list if s.player == player_name]

    if not player_stats:
        print(f"No stats found for player: {player_name}")
        return pd.DataFrame()

    rows = []
    for s in player_stats:
        rows.append({
            "season":     s.season,
            "player_id":  s.player_id,
            "player":     s.player,
            "position":   s.position,
            "team":       s.team,
            "conference": s.conference,
            "category":   s.category,
            "stat_type":  s.stat_type,
            "stat":       s.stat
        })

    long_df = pd.DataFrame(rows)

    long_df["col_name"] = long_df["category"] + "_" + long_df["stat_type"].str.replace(" ", "_")

    wide_df = long_df.pivot_table(
        index=["season", "player_id", "player", "position", "team", "conference"],
        columns="col_name",
        values="stat",
        aggfunc="first"
    ).reset_index()

    wide_df.columns.name = None

    return wide_df

In [133]:
sorted_data = parse_cfbd_player_stats(raw_data, "Trevon Diggs")

sorted_data.to_csv('data/CB/TDiggs_CB.csv', index=False)

sorted_data

,season,player_id,player,position,team,conference,defensive_PD,defensive_QB_HUR,defensive_SACKS,defensive_SOLO,...,fumbles_REC,interceptions_AVG,interceptions_INT,interceptions_TD,interceptions_YDS,kickReturns_AVG,kickReturns_LONG,kickReturns_NO,kickReturns_TD,kickReturns_YDS
0,2019,4040966,Trevon Diggs,DB,Alabama,SEC,8,0,0,20,...,2,26.3,3,1,79,19.5,36,6,0,117
